In [15]:
import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from imblearn.over_sampling import SMOTE

# Chargement des données nettoyées (créées dans EDA.ipynb)
try:
    df = pd.read_csv("flight_delays_cleaned_sample.csv")
    print(f"✅ Data loeaded : {df.shape[0]} lines and {df.shape[1]} columns.")
    # Aperçu pour vérifier que tout est là
    print(df.head(3))
except FileNotFoundError:
    print("❌ Error : File 'flight_delays_cleaned_sample.csv' doesn't exist.")

✅ Data loeaded : 98519 lines and 12 columns.
   MONTH  DAY  DAY_OF_WEEK AIRLINE_x                 airline_name  \
0      4    7            2        EV  Atlantic Southeast Airlines   
1      1   24            6        AS         Alaska Airlines Inc.   
2      7    8            3        WN       Southwest Airlines Co.   

  ORIGIN_AIRPORT DESTINATION_AIRPORT  SCHEDULED_DEPARTURE  DISTANCE  \
0            FWA                 DTW                 1340       128   
1            LAS                 SEA                 1910       867   
2            OAK                 SEA                  630       672   

   DEPARTURE_DELAY  delayed  HOUR  
0             -5.0        0    13  
1            -12.0        0    19  
2             -4.0        0     6  


In [16]:
# La cible est 'delayed' (1 ou 0)
y = df["delayed"]

# Les features (X) : on retire la cible et les infos inutiles
# On retire 'DEPARTURE_DELAY' car c'est la réponse (triche)
# On retire 'AIRLINE_x' car on utilise 'airline_name'
cols_to_drop = ["delayed", "DEPARTURE_DELAY", "AIRLINE_x"]
X = df.drop(columns=cols_to_drop, errors='ignore')

print(f"Dimensions of X : {X.shape}")
print("Columns used for prediction :")
print(list(X.columns))

Dimensions of X : (98519, 9)
Columns used for prediction :
['MONTH', 'DAY', 'DAY_OF_WEEK', 'airline_name', 'ORIGIN_AIRPORT', 'DESTINATION_AIRPORT', 'SCHEDULED_DEPARTURE', 'DISTANCE', 'HOUR']


In [17]:
# Identification automatique des colonnes
numeric_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_cols = X.select_dtypes(include=['object']).columns.tolist()

print(f"Numerical Variables (to scaler) : {numeric_cols}")
print(f"Categorical Variables (to encode) : {categorical_cols}")

# Création du "Robot" de transformation
preprocessor = ColumnTransformer(
    transformers=[
        # StandardScaler met les chiffres sur la même échelle (Mois, Distance, Heure)
        ('num', StandardScaler(), numeric_cols),

        # OneHotEncoder transforme les aéroports et compagnies en 0 et 1
        # handle_unknown='ignore' est VITAL : si un petit aéroport est dans le test
        # mais pas dans le train, le modèle ne plantera pas.
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_cols)
    ],
    verbose_feature_names_out=False # Pour avoir des noms de colonnes plus propres
)

print("Preprocessor configured.")

Numerical Variables (to scaler) : ['MONTH', 'DAY', 'DAY_OF_WEEK', 'SCHEDULED_DEPARTURE', 'DISTANCE', 'HOUR']
Categorical Variables (to encode) : ['airline_name', 'ORIGIN_AIRPORT', 'DESTINATION_AIRPORT']
Preprocessor configured.


In [18]:
# 1. Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# 2. Fit & Transform
# On apprend les échelles sur le TRAIN seulement (pour éviter la fuite de données)
print("Transformation of training data (can take several seconds)...")
X_train_encoded = preprocessor.fit_transform(X_train)

# On applique juste la transformation sur le TEST
print("Transformation of test data...")
X_test_encoded = preprocessor.transform(X_test)

print(f"Shape of the data after encoding: {X_train_encoded.shape}")
# Note : Le nombre de colonnes aura augmenté (probablement > 600) à cause des aéroports. C'est normal.

Transformation of training data (can take several seconds)...
Transformation of test data...
Shape of the data after encoding: (78815, 1173)


In [19]:
# Equilibrage des classes avec SMOTE
print(f"Distribution before SMOTE : {np.bincount(y_train)} (0=On time, 1=Delay)")

# Initialisation de SMOTE
smote = SMOTE(random_state=42)

# Application (Peut prendre 10-30 secondes selon votre PC)
print("Generation of synthetic samples with SMOTE (can take some time)...")
X_train_resampled, y_train_resampled = smote.fit_resample(X_train_encoded, y_train)

print(f"Distribution after SMOTE : {np.bincount(y_train_resampled)}")
print("Classes balanced.")

Distribution before SMOTE : [64772 14043] (0=On time, 1=Delay)
Generation of synthetic samples with SMOTE (can take some time)...
Distribution after SMOTE : [64772 64772]
Classes balanced.


In [20]:
# Récupération des noms de features (Astuce technique pour l'interprétabilité)
try:
    feature_names = preprocessor.get_feature_names_out()
except:
    # Fallback si l'extraction automatique échoue
    feature_names = [f"feat_{i}" for i in range(X_train_resampled.shape[1])]

print(f"Name of extracted features : {len(feature_names)}")

# Création du dictionnaire à sauvegarder
data_to_save = {
    'X_train': X_train_resampled,
    'y_train': y_train_resampled,
    'X_test': X_test_encoded,
    'y_test': y_test,
    'feature_names': feature_names
}

# Sauvegarde sur le disque
joblib.dump(data_to_save, 'processed_data.pkl')

print("Done ! File 'processed_data.pkl' Success.")

Name of extracted features : 1173
Done ! File 'processed_data.pkl' Success.
